# FL Phase A — Baseline Reproduction (Kaggle, GPU)

Thesis baseline (MNIST + Credit-Card x committee_consensus + trust_based) reproduce করা।
Target branch: `het-aware-trust`।

**চালানোর আগে:**
1. Settings (ডান পাশ) → Accelerator → **GPU T4**।
2. Add-ons → Secrets → `KAGGLE_API_TOKEN` নামে secret যোগ করুন (value = Kaggle → Settings → API → Create New Token), আর এই notebook-এ secret-টা attach করুন।
3. Cell গুলো উপর থেকে নিচে একটা একটা করে Run করুন। কোনো cell-এ লাল error এলে থামুন, error-টা copy করে Saimoon-কে পাঠান।

In [ ]:
# Cell 1 — environment check (কিছু install করে না, শুধু দেখে)
import sys, importlib.util
print('python:', sys.version.split()[0])
for pkg in ['tensorflow', 'keras', 'sklearn', 'pandas', 'numpy', 'fastapi', 'pydantic', 'kaggle']:
    print(pkg, '=>', importlib.util.find_spec(pkg) is not None)
try:
    import tensorflow as tf
    print('TF version:', tf.__version__)
except Exception as e:
    print('TF import FAILED:', repr(e))

In [ ]:
# Cell 2 — clone repo branch (already থাকলে skip)
import os
if not os.path.isdir('Federated-Learning-using-Blockchain'):
    !git clone -b het-aware-trust https://github.com/saimoon-oman/Federated-Learning-using-Blockchain.git
else:
    print('repo already present')
%cd Federated-Learning-using-Blockchain
!git log --oneline -3

In [ ]:
# Cell 3 — installs, SPLIT in two (একসাথে দিলে tf-privacy পুরো run-টাই fail করায়)
# Part 1: safe packages — এগুলো অবশ্যই OK হতে হবে
!pip install -q fastapi uvicorn pydantic python-multipart 2>&1 | tail -n 2
# Part 2: tf-privacy — Python 3.12-এ FAIL HOBEI, এটা expected; Phase A আটকাবে না
!pip install -q "tensorflow-privacy==0.9.0" 2>&1 | tail -n 2
import importlib
for pkg in ['fastapi', 'uvicorn', 'multipart', 'pydantic', 'sklearn', 'pandas']:
    try:
        importlib.import_module(pkg)
        print(pkg, 'OK')
    except Exception as e:
        print(pkg, 'FAILED:', repr(e), '<== থামুন, error Saimoon-কে পাঠান')
try:
    importlib.import_module('tensorflow_privacy')
    print('tensorflow_privacy OK (DP arm available)')
except Exception:
    print('tensorflow_privacy MISSING (expected on Python 3.12) — Phase A এগোবে, DP arm পরে')

In [ ]:
# Cell 4 — Kaggle auth (Secret থেকে, কোথাও paste/commit হয় না) + dataset download
import os
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret('KAGGLE_API_TOKEN')
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
with open(os.path.expanduser('~/.kaggle/access_token'), 'w') as f:
    f.write(token)
os.chmod(os.path.expanduser('~/.kaggle/access_token'), 0o600)
print('kaggle auth file written (VM only, not committed)')
!kaggle datasets download -d nelgiriyewithana/credit-card-fraud-detection-dataset-2023 -p python/model/ --unzip
!ls -lh python/model/creditcard_2023.csv

In [ ]:
# Cell 5 — CSV sanity check (column অবশ্যই id,V1..V28,Amount,Class হতে হবে)
import pandas as pd
df = pd.read_csv('python/model/creditcard_2023.csv', nrows=3)
print(list(df.columns))
assert list(df.columns)[:3] == ['id', 'V1', 'V2'] and list(df.columns)[-2:] == ['Amount', 'Class'], 'COLUMN MISMATCH — থামুন, Saimoon-কে জানান'
print('columns OK')

In [ ]:
# Cell 6 — Phase A run (30-60 min লাগতে পারে; browser বন্ধ করবেন না)
!python kaggle/phase_a_baseline.py

In [ ]:
# Cell 7 — ফলাফল summary (thesis Fig 4.1-4.4-এর সাথে তুলনার জন্য)
import glob, json
fp = sorted(glob.glob('kaggle/phase_a_results_*.json'))[-1]
print('reading', fp)
res = json.load(open(fp))
for key, val in res.items():
    if not val.get('ok'):
        print(key, '=> FAILED:', val.get('error'))
        continue
    print(f"\n=== {key} (total {val['total_s']}s) ===")
    for r in val['rounds']:
        m = r['metrics']
        print(f"round {r['round']}: acc={m['accuracy']:.4f} prec={m['precision']:.4f} rec={m['recall']:.4f} f1={m['f1']:.4f} ({r['seconds']}s)")

In [ ]:
# Cell 8 — quick plot (accuracy vs round, 4 config)
import glob, json
import matplotlib.pyplot as plt
fp = sorted(glob.glob('kaggle/phase_a_results_*.json'))[-1]
res = json.load(open(fp))
for key, val in res.items():
    if val.get('ok'):
        xs = [r['round'] for r in val['rounds']]
        ys = [r['metrics']['accuracy'] for r in val['rounds']]
        plt.plot(xs, ys, marker='o', label=key)
plt.xlabel('round'); plt.ylabel('accuracy'); plt.legend(fontsize=7); plt.grid(True)
plt.savefig('kaggle/phase_a_accuracy.png', dpi=150)
plt.show()
print('saved kaggle/phase_a_accuracy.png — এই PNG + JSON দুটোই Saimoon-কে পাঠান')